In [1]:
# 0. INSTALLATION ET IMPORTS
!pip install xgboost -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. CHARGEMENT DU FICHIER (upload manuel)
print("Veuillez téléverser le fichier dataset_heart.csv")
uploaded = files.upload()
# Récupérer le nom du fichier chargé
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

# Nettoyer les noms de colonnes (enlever les espaces)
df.columns = df.columns.str.strip()
print("Dataset chargé :", df.shape)
print(df.head())

# EXERCICE 1 : EDA
print("\n EXERCICE 1 : Analyse exploratoire ")
print("Informations :")
print(df.info())
print("\nStatistiques :")
print(df.describe())

# Visualisation
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
sns.countplot(data=df, x='heart disease')
plt.title("Distribution de la cible (heart disease)")
plt.subplot(1,2,2)
df.corr()['heart disease'].sort_values().plot(kind='barh')
plt.title("Corrélation avec la cible")
plt.tight_layout()
plt.show()

# Séparation features / cible
X = df.drop('heart disease', axis=1)
y = df['heart disease']

# Transformer la cible de {1,2} en {0,1} si nécessaire
if set(y.unique()) == {1,2}:
    y = y - 1
    print("Cible transformée : 1->0 (absence), 2->1 (présence)")

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train : {X_train.shape}, Test : {X_test.shape}")

# Standardisation (pour LR et SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# EXERCICE 2 : Logistic Regression sans GridSearch
print("\n EXERCICE 2 : Logistic Regression (sans GridSearch) ")
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy : {acc_lr:.4f}")
print(classification_report(y_test, y_pred_lr))

# EXERCICE 3 : Logistic Regression avec GridSearch
print("\n EXERCICE 3 : Logistic Regression (avec GridSearch) ")
param_grid_lr = {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}
grid_lr = GridSearchCV(LogisticRegression(random_state=42, max_iter=1000), param_grid_lr, cv=5, scoring='accuracy')
grid_lr.fit(X_train_scaled, y_train)
print(f"Meilleurs paramètres : {grid_lr.best_params_}")
print(f"Accuracy validation : {grid_lr.best_score_:.4f}")
y_pred_lr_gs = grid_lr.best_estimator_.predict(X_test_scaled)
print(f"Accuracy test : {accuracy_score(y_test, y_pred_lr_gs):.4f}")

# EXERCICE 4 : SVM sans GridSearch
print("\n EXERCICE 4 : SVM (sans GridSearch) ")
svm = SVC(kernel='rbf', C=1, gamma='scale', random_state=42)
svm.fit(X_train_scaled, y_train)
y_pred_svm = svm.predict(X_test_scaled)
acc_svm = accuracy_score(y_test, y_pred_svm)
print(f"Accuracy : {acc_svm:.4f}")
print(classification_report(y_test, y_pred_svm))

# EXERCICE 5 : SVM avec GridSearch
print("\n EXERCICE 5 : SVM (avec GridSearch) ")
param_grid_svm = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf'], 'gamma': ['scale', 'auto', 0.1]}
grid_svm = GridSearchCV(SVC(random_state=42), param_grid_svm, cv=5, scoring='accuracy')
grid_svm.fit(X_train_scaled, y_train)
print(f"Meilleurs paramètres : {grid_svm.best_params_}")
print(f"Accuracy validation : {grid_svm.best_score_:.4f}")
y_pred_svm_gs = grid_svm.best_estimator_.predict(X_test_scaled)
print(f"Accuracy test : {accuracy_score(y_test, y_pred_svm_gs):.4f}")

# EXERCICE 6 : XGBoost sans GridSearch
print("\n EXERCICE 6 : XGBoost (sans GridSearch) ")
# Justification : learning_rate=0.1 (valeur par défaut efficace), n_estimators=100 (compromis), max_depth=5 (évite overfitting)
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)  # XGBoost supporte les données non standardisées
y_pred_xgb = xgb.predict(X_test)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
print(f"Accuracy : {acc_xgb:.4f}")
print(classification_report(y_test, y_pred_xgb))

# EXERCICE 7 : XGBoost avec GridSearch
print("\n EXERCICE 7 : XGBoost (avec GridSearch) ")
param_grid_xgb = {
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0]
}
grid_xgb = GridSearchCV(XGBClassifier(random_state=42, eval_metric='logloss'),
                        param_grid_xgb, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_xgb.fit(X_train, y_train)
print(f"Meilleurs paramètres : {grid_xgb.best_params_}")
print(f"Accuracy validation : {grid_xgb.best_score_:.4f}")
y_pred_xgb_gs = grid_xgb.best_estimator_.predict(X_test)
print(f"Accuracy test : {accuracy_score(y_test, y_pred_xgb_gs):.4f}")

# RÉCAPITULATIF FINAL
print("\n" + "="*60)
print("RÉSULTATS DES MODÈLES (accuracy sur le test)")
print("="*60)
results = {
    "LogReg sans GS": acc_lr,
    "LogReg avec GS": accuracy_score(y_test, y_pred_lr_gs),
    "SVM sans GS": acc_svm,
    "SVM avec GS": accuracy_score(y_test, y_pred_svm_gs),
    "XGBoost sans GS": acc_xgb,
    "XGBoost avec GS": accuracy_score(y_test, y_pred_xgb_gs)
}
for name, acc in results.items():
    print(f"{name:20} : {acc:.4f}")
best = max(results, key=results.get)
print(f"\n Meilleur modèle : {best} avec {results[best]:.4f}")


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'google.colab'